# WTDF - Modeling

This notebook has been updated to match the current WTDF modeling stack:

- canonical state labels now come from preprocessing
- experiment-specific binary targets are derived with `WindFarmSplitter.create_binary_target_from_state(...)`
- splitting uses the current splitter API
- model fitting / threshold tuning / evaluation use the updated `WindFaultTrainer`
- optional threshold sweep diagnostics use `wtfd.models.metrics`

The notebook is intentionally structured as a thin experiment runner and analysis notebook, with core training logic living in modules.

In [ ]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from wtfd.models.splitter import WindFarmSplitter
from wtfd.models.trainer import WindFaultTrainer
from wtfd.models.metrics import build_threshold_sweep_table

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

## 1. Experiment configuration

In [ ]:
# Project / data paths
PROJ_ROOT = Path("..")
MASTER_DATASET_PATH = PROJ_ROOT / "data" / "processed" / "master_dataset.parquet"

# Experiment settings
RANDOM_STATE = 42
N_SPLITS = 5

HORIZON_HOURS = 72              # Supported: 24, 48, 72
INCLUDE_EVENT = False           # If True, event_occurring rows are treated as positive
DROP_BUFFER = True              # Recommended: True
KEEP_ONLY_RELEVANT_ROWS = True  # Recommended: True

# Split strategy
SPLIT_STRATEGY = "event_time"   # Options: "event_time", "group_time", "global_time", "group_holdout"
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15

# Model comparison
MODEL_TYPES = ["xgboost", "rf", "logistic"]
THRESHOLD_OBJECTIVE = "f1"      # Supported by updated metrics/trainer: f1, precision, recall, balanced_accuracy, specificity

# Visualization
TOP_N_FEATURES = 15

print(f"Master dataset path: {MASTER_DATASET_PATH}")
print(f"Horizon: {HORIZON_HOURS}h | Include event rows: {INCLUDE_EVENT}")
print(f"Split strategy: {SPLIT_STRATEGY}")
print(f"Models: {MODEL_TYPES}")

## 2. Load canonical processed dataset and derive the modeling target

The processed master dataset retains canonical state labels (`state_label`, `state_name`) plus a convenience `target`.  
For modeling, we now derive the *experiment-specific* binary target from the canonical state labels using the splitter.


In [ ]:
if not MASTER_DATASET_PATH.exists():
    raise FileNotFoundError(f"Master dataset not found: {MASTER_DATASET_PATH}")

print("Loading canonical processed dataset...")
df = pd.read_parquet(MASTER_DATASET_PATH)

print(f"Loaded dataset shape: {df.shape}")
display(df.head())

splitter = WindFarmSplitter(n_splits=N_SPLITS, random_state=RANDOM_STATE)

modeling_df = splitter.create_binary_target_from_state(
    df=df,
    horizon_hours=HORIZON_HOURS,
    include_event=INCLUDE_EVENT,
    drop_buffer=DROP_BUFFER,
    target_col="target",
    keep_only_relevant_rows=KEEP_ONLY_RELEVANT_ROWS,
)

del df
gc.collect()

print(f"Modeling dataset shape after horizon filtering: {modeling_df.shape}")

In [ ]:
summary_df = pd.DataFrame(
    {
        "rows": [len(modeling_df)],
        "unique_farms": [modeling_df["farm_id"].nunique() if "farm_id" in modeling_df.columns else np.nan],
        "unique_turbines": [modeling_df["asset_id"].nunique() if "asset_id" in modeling_df.columns else np.nan],
        "unique_events": [modeling_df["event_id"].nunique() if "event_id" in modeling_df.columns else np.nan],
        "positive_rate": [modeling_df["target"].mean()],
    }
)

state_counts = (
    modeling_df["state_name"]
    .value_counts(dropna=False)
    .rename_axis("state_name")
    .reset_index(name="row_count")
)

target_counts = (
    modeling_df["target"]
    .value_counts(dropna=False)
    .rename_axis("target")
    .reset_index(name="row_count")
    .sort_values("target")
    .reset_index(drop=True)
)

display(summary_df)
display(state_counts)
display(target_counts)

## 3. Create train / validation / test splits

Recommended primary strategy: event-level chronological split, which keeps entire events together while respecting time order.


In [ ]:
if SPLIT_STRATEGY == "event_time":
    train_df, val_df, test_df = splitter.get_event_level_time_split(
        modeling_df,
        train_size=TRAIN_SIZE,
        val_size=VAL_SIZE,
        test_size=TEST_SIZE,
    )
elif SPLIT_STRATEGY == "group_time":
    train_df, val_df, test_df = splitter.get_grouped_time_split_by_turbine(
        modeling_df,
        train_size=TRAIN_SIZE,
        val_size=VAL_SIZE,
        test_size=TEST_SIZE,
    )
elif SPLIT_STRATEGY == "global_time":
    train_df, val_df, test_df = splitter.get_train_val_test_split_by_time(
        modeling_df,
        train_size=TRAIN_SIZE,
        val_size=VAL_SIZE,
        test_size=TEST_SIZE,
    )
elif SPLIT_STRATEGY == "group_holdout":
    dev_df, test_df = splitter.get_group_aware_train_test_split(
        modeling_df,
        test_size=TEST_SIZE,
    )
    # Reuse the grouped time split within the development data to create train / validation
    train_frac_of_dev = TRAIN_SIZE / (TRAIN_SIZE + VAL_SIZE)
    val_frac_of_dev = VAL_SIZE / (TRAIN_SIZE + VAL_SIZE)
    train_df, val_df, _ = splitter.get_grouped_time_split_by_turbine(
        dev_df,
        train_size=train_frac_of_dev,
        val_size=val_frac_of_dev,
        test_size=0.0,
    )
else:
    raise ValueError(f"Unsupported SPLIT_STRATEGY: {SPLIT_STRATEGY}")

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "positives": int(train_df["target"].sum()),
            "positive_rate": float(train_df["target"].mean()),
            "unique_turbines": train_df["asset_id"].nunique(),
            "unique_events": train_df["event_id"].nunique() if "event_id" in train_df.columns else np.nan,
            "min_time": train_df["time_stamp"].min() if "time_stamp" in train_df.columns else pd.NaT,
            "max_time": train_df["time_stamp"].max() if "time_stamp" in train_df.columns else pd.NaT,
        },
        {
            "split": "validation",
            "rows": len(val_df),
            "positives": int(val_df["target"].sum()),
            "positive_rate": float(val_df["target"].mean()),
            "unique_turbines": val_df["asset_id"].nunique(),
            "unique_events": val_df["event_id"].nunique() if "event_id" in val_df.columns else np.nan,
            "min_time": val_df["time_stamp"].min() if "time_stamp" in val_df.columns else pd.NaT,
            "max_time": val_df["time_stamp"].max() if "time_stamp" in val_df.columns else pd.NaT,
        },
        {
            "split": "test",
            "rows": len(test_df),
            "positives": int(test_df["target"].sum()),
            "positive_rate": float(test_df["target"].mean()),
            "unique_turbines": test_df["asset_id"].nunique(),
            "unique_events": test_df["event_id"].nunique() if "event_id" in test_df.columns else np.nan,
            "min_time": test_df["time_stamp"].min() if "time_stamp" in test_df.columns else pd.NaT,
            "max_time": test_df["time_stamp"].max() if "time_stamp" in test_df.columns else pd.NaT,
        },
    ]
)

display(split_summary)

In [ ]:
X_train, y_train = splitter.prepare_xy(train_df, target_col="target")
X_val, y_val = splitter.prepare_xy(val_df, target_col="target")
X_test, y_test = splitter.prepare_xy(test_df, target_col="target")

feature_names = X_train.columns.tolist()

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Number of modeling features: {len(feature_names)}")

## 4. Model tournament

Each candidate model is:
1. fit on the training set
2. threshold-tuned on the validation set
3. evaluated on train / validation / test


In [ ]:
tournament_rows = []
detailed_results = {}

for model_type in MODEL_TYPES:
    print(f"\n{'=' * 80}")
    print(f"Training model: {model_type}")
    print(f"{'=' * 80}")

    trainer = WindFaultTrainer(
        model_type=model_type,
        random_state=RANDOM_STATE,
    )

    tuned_threshold = trainer.fit_and_tune(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        optimize_for=THRESHOLD_OBJECTIVE,
    )

    train_metrics = trainer.evaluate_detailed(X_train, y_train)
    val_metrics = trainer.evaluate_detailed(X_val, y_val)
    test_metrics = trainer.evaluate_detailed(X_test, y_test)

    tournament_rows.append(
        {
            "model_type": model_type,
            "threshold_objective": THRESHOLD_OBJECTIVE,
            "tuned_threshold": tuned_threshold,
            "train_precision": train_metrics["precision"],
            "train_recall": train_metrics["recall"],
            "train_f1": train_metrics["f1"],
            "train_roc_auc": train_metrics["roc_auc"],
            "train_pr_auc": train_metrics["pr_auc"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "val_roc_auc": val_metrics["roc_auc"],
            "val_pr_auc": val_metrics["pr_auc"],
            "test_precision": test_metrics["precision"],
            "test_recall": test_metrics["recall"],
            "test_f1": test_metrics["f1"],
            "test_roc_auc": test_metrics["roc_auc"],
            "test_pr_auc": test_metrics["pr_auc"],
            "test_balanced_accuracy": test_metrics["balanced_accuracy"],
            "test_specificity": test_metrics["specificity"],
            "test_tp": test_metrics["tp"],
            "test_fp": test_metrics["fp"],
            "test_tn": test_metrics["tn"],
            "test_fn": test_metrics["fn"],
        }
    )

    detailed_results[model_type] = {
        "trainer": trainer,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
    }

results_df = pd.DataFrame(tournament_rows).sort_values(
    by=["test_pr_auc", "test_recall", "test_f1"],
    ascending=[False, False, False],
).reset_index(drop=True)

display(results_df)

In [ ]:
best_model_type = results_df.iloc[0]["model_type"]
best_bundle = detailed_results[best_model_type]
best_trainer = best_bundle["trainer"]

print(f"Winning model: {best_model_type}")
print(f"Tuned threshold: {best_trainer.best_threshold:.6f}")

best_test_metrics = best_bundle["test_metrics"]
metric_snapshot = pd.DataFrame(
    [
        {
            "model_type": best_model_type,
            "precision": best_test_metrics["precision"],
            "recall": best_test_metrics["recall"],
            "f1": best_test_metrics["f1"],
            "roc_auc": best_test_metrics["roc_auc"],
            "pr_auc": best_test_metrics["pr_auc"],
            "balanced_accuracy": best_test_metrics["balanced_accuracy"],
            "specificity": best_test_metrics["specificity"],
            "threshold": best_test_metrics["threshold"],
            "tp": best_test_metrics["tp"],
            "fp": best_test_metrics["fp"],
            "tn": best_test_metrics["tn"],
            "fn": best_test_metrics["fn"],
        }
    ]
)

display(metric_snapshot)

## 5. Feature importance / coefficient inspection for the winning model

In [ ]:
importance_df = best_trainer.get_feature_importance(feature_names).copy()
display(importance_df.head(TOP_N_FEATURES))

plot_df = importance_df.head(TOP_N_FEATURES).iloc[::-1].copy()

value_col = "importance" if "importance" in plot_df.columns else "coefficient"
label = "Importance" if value_col == "importance" else "Coefficient"

plt.figure(figsize=(10, 6))
plt.barh(plot_df["feature"], plot_df[value_col])
plt.title(f"Top {TOP_N_FEATURES} Features - {best_model_type}")
plt.xlabel(label)
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 6. Threshold sweep diagnostics for the winning model

This is useful when you want to compare precision / recall tradeoffs more explicitly than a single tuned threshold allows.


In [ ]:
best_test_probs = best_trainer.predict_proba(X_test)
threshold_df = build_threshold_sweep_table(
    y_true=y_test,
    y_prob=best_test_probs,
)

display(
    threshold_df.sort_values(
        by=["f1", "recall", "precision"],
        ascending=[False, False, False],
    ).head(15)
)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(threshold_df["threshold"], threshold_df["precision"], label="Precision")
plt.plot(threshold_df["threshold"], threshold_df["recall"], label="Recall")
plt.plot(threshold_df["threshold"], threshold_df["f1"], label="F1")
plt.axvline(best_trainer.best_threshold, linestyle="--", label="Tuned threshold")
plt.title(f"Threshold Sweep - {best_model_type}")
plt.xlabel("Threshold")
plt.ylabel("Metric value")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Timeline-style inspection for one positive test event / turbine

In [ ]:
test_scored_df = test_df.copy()
test_scored_df["pred_prob"] = best_test_probs
test_scored_df["pred_label"] = (test_scored_df["pred_prob"] >= best_trainer.best_threshold).astype(int)

candidate_df = test_scored_df[test_scored_df["target"] == 1].copy()

if candidate_df.empty:
    print("No positive rows found in the test split for timeline inspection.")
else:
    if "event_id" in candidate_df.columns:
        selected_event_id = candidate_df["event_id"].iloc[0]
        plot_df = (
            test_scored_df[test_scored_df["event_id"] == selected_event_id]
            .sort_values("time_stamp")
            .copy()
        )
        title_suffix = f"event_id={selected_event_id}"
    else:
        selected_turbine = candidate_df["asset_id"].iloc[0]
        plot_df = (
            test_scored_df[test_scored_df["asset_id"] == selected_turbine]
            .sort_values("time_stamp")
            .copy()
        )
        title_suffix = f"asset_id={selected_turbine}"

    display(
        plot_df[
            ["time_stamp", "farm_id", "asset_id", "event_id", "state_name", "target", "pred_prob", "pred_label"]
        ].head(25)
    )

    plt.figure(figsize=(16, 5))
    plt.plot(plot_df["time_stamp"], plot_df["pred_prob"], label="Predicted probability")
    plt.axhline(best_trainer.best_threshold, linestyle="--", label="Tuned threshold")

    if "target" in plot_df.columns:
        positive_mask = plot_df["target"] == 1
        plt.scatter(
            plot_df.loc[positive_mask, "time_stamp"],
            plot_df.loc[positive_mask, "pred_prob"],
            label="Positive rows",
            s=30,
        )

    plt.title(f"Prediction Timeline - {best_model_type} ({title_suffix})")
    plt.xlabel("Time")
    plt.ylabel("Predicted probability")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 8. Optional next steps

Good follow-on experiments for this notebook:
- compare 24h vs 48h vs 72h horizons
- compare `include_event=False` vs `include_event=True`
- compare event-level time split against grouped holdout split
- export `results_df`, `importance_df`, and `threshold_df` for report-ready tables
